# 1. Data Acquisition

First stage of the pipeline. It reads the raw Colombo Stock Exchange archive and the
EM-DAT disaster export from `data/raw/`, fetches the macroeconomic and global controls
live, and caches four tables to `artifacts/tables/`: `market`, `disasters`, `macro` and
`sp500`. Nothing downstream reads a raw file again.

This is the only stage that depends on a network connection, and its sources are live and
unpinned. It therefore ships without stored cell outputs: re running it can return revised
figures, which would move every downstream number, so the cached artifacts in
`artifacts/tables/` are the record of what this stage produced. See
[`docs/data_sources.md`](../docs/data_sources.md) for the source by source detail and
[`docs/notebooks/01_data_acquisition.md`](../docs/notebooks/01_data_acquisition.md) for
this notebook's own documentation.


## 1.1 Environment and paths

Loads `_shared.py` (paths, the artifact cache helpers, the target definitions) and applies the thesis figure style, then prints the artifact cache so it is visible which upstream stage produced these inputs and when.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "_shared.py").exists() else Path.cwd() / "notebooks"))
from _shared import *  # noqa: F401,F403  paths, artifact cache, target bounds

from src.visualization import result_figures as fx
fx.apply_thesis_style()

print("stage inputs available in the artifact cache:")
print(artifact_status().to_string(index=False) if len(artifact_status()) else "  (none yet)")


## 1.2 Source validation and data inventory

Before writing a single line of pipeline code, every claimed data source and external library citation in the project's supporting documents (a "Resource Matrix", a "Detail Guide", and a "Master Expert Panel Blueprint") was checked against reality, either by direct inspection of the files actually supplied, or by live lookups against the source itself. Nothing below is asserted from memory.

### 1.1 Data files actually supplied, as inspected

| File | Usable? | Content | Real date range (verified) | Role |
|---|---|---|---|---|
| `07Market Indices - Daily.xls` (sheet "Index") |  | Daily ASPI, Milanka, S&P SL20 + ~20 sector indices | **2-Jan-1985  28-Jun-2023** (5,597 rows ≥2000-01-01) | Primary ASPI price series (Y1) |
| 24 yearly `<year> [Dd]ata.xls[x]` files (2000, 2023) |  (23/24) | Per-security daily closing price + volume, **three distinct real report layouts** across the years | 2001  Q1-2023 volume; 2000 is price-only, no volume field at all | Aggregated market-wide volume (Y2 baseline) |
| `public_emdat_custom_request_2026-02-09_...xlsx` |  | Real EM-DAT export, standard 47-column public schema, 86 disaster records for Sri Lanka | includes the 2025 "Ditwah" storm record | Primary disaster source (exogenous features) |
| `market-capitalization-Oct 12.csv` |  supplementary | Single-date company market-cap snapshot | one date only | Not wired into the core pipeline, optional sector-mapping enrichment |
| `2024 ADB Asia SME Monitor - SRI.xlsx` |  not relevant | MSME definitions/financing tables | 2019, 2023 | **Not** a macro-control source (no GDP/inflation/FX/rate series), does not fill that gap |
| `CSE All-Share Historical Data.csv` |  dead | Header row only, 55 bytes, zero data rows |,  | Discarded |

**Critical, load-bearing finding:** the local archive's real coverage stops at **28-Jun-2023** (price) / **31-Mar-2023** (volume). The thesis's own flagship example event, the **2025 "Ditwah" storm**, shown in the thesis's Fig. 1, falls entirely outside this window. See §6 below for how this was handled (spoiler: not silently papered over).

### 1.2 Thesis Table 4 (§3.2.2) vs. what this notebook actually uses

| Thesis Table 4 variable | Thesis-named source | What this notebook uses |
|---|---|---|
| ASPI % change (Y1) | "Colombo Stock Exchange (CSE)" | `07Market Indices - Daily.xls`, real CSE-format archive |
| Volume Crash Magnitude (Y2) | CSE | Aggregated per-security volume from the 24 yearly files, same archive |
| Market Recovery Days (Y3) | CSE | Derived from the same ASPI series |
| Disaster Financial Damage / Population Affected / Disaster Type | EM-DAT | Real EM-DAT export, mapped 1:1 |
| Macroeconomic Stability | **"Central Bank of Sri Lanka; World Bank"** (thesis's own words) | World Bank via `wbgapi`, the thesis-named World Bank source; CBSL's higher-frequency series (CCPI/FX/rate) undocumented gap, see §11 |
| Global Market Conditions | **"Yahoo Finance / Bloomberg"** (thesis's own words) | `yfinance`, S&P 500 (`^GSPC`), thesis-named source, verified live and working |

No source here was substituted from outside what the thesis itself names in Table 4 / §3.3.1.

### 1.3 External citation fact-checks (Resource Matrix / Detail Guide / Blueprint documents)

| Claim | Status | Finding |
|---|---|---|
| `yfinance` ticker `^CSE` exists |  real | Confirmed live: Yahoo lists "COLOMBO IND ALL SHS (^CSE)" |
| `^CSE`'s data is queryable for recent dates |  **false** | Verified via 3 independent methods (yfinance `.download()`, `.history()`, Yahoo's raw chart API directly): the feed stopped updating **~2019** (`regularMarketTime`Jun-2019, zero rows for any 2020+ range). The ticker *existing as a page* is not the same as it having a *live feed*, an important distinction this notebook's own first pass got wrong before re-checking (see §6). |
| `wbgapi` (World Bank Python package) |  real | `tgherzog/wbgapi`, confirmed working live |
| EM-DAT free academic CSV export |  partially correct | Real, but primary export format is **Excel (.xlsx)**, not CSV as claimed; CSV only via the separate EM-VIEW dashboard |
| `nickkunz/smogn`, `paobranco/ImbalancedLearningRegression` |  real | Both confirmed to exist on GitHub |
| `shap` package |  real | Canonical org now `shap/shap` (moved from `slundberg/shap`) |
| **`Bae-Youn/eventstudy`** (cited in two of the supplied documents) |  **fabricated** | No such repository or author exists anywhere. Real package: `LemaireJean-Baptiste/eventstudy` (PyPI `eventstudy`) |
| EM-DAT "Total Damage, Adjusted" column methodology |  verified | doc.emdat.be confirms it applies **OECD CPI** to inflate raw damage relative to Start Year, i.e. it already satisfies a PPP/inflation-adjustment requirement natively (see §5) |

In [ ]:
# Paths, RANDOM_STATE and the artifact helpers now come from _shared; this cell only
# reports the environment so the executed notebook records what produced its numbers.
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

print(f"repo root : {REPO_ROOT}")
print(f"raw data  : {RAW_DATA_DIR}  ({len(list(RAW_DATA_DIR.glob('*'))) if RAW_DATA_DIR.exists() else 0} files)")
print(f"artifacts : {ARTIFACT_DIR}")
print(f"figures   : {FIGURE_DIR}")
print(f"seed      : {RANDOM_STATE}")


## 1.3 Acquiring the raw series

Every loader below reads the **real files described in §1.1**. Each was verified during development by direct inspection of its output against the source workbooks (three distinct per-security report layouts required a header-text-driven parser rather than a fixed-column-index one, column order and even column *count* drift year to year in this real archive; see `src/data/cse_market_data.py` docstrings for the full provenance notes).

**Automated test coverage is limited to `src/features/feature_engineering.py` and `src/training/time_aware_smogn.py`** (`tests/test_feature_engineering.py`, `tests/test_time_aware_smogn.py`). The loaders themselves have no unit tests, recorded here as outstanding work rather than implied to exist.

### 1.3.1 Market series

Parses every yearly per-security workbook and the ASPI index sheet into one daily
`date, aspi_close, trading_volume` series. The loader locates headers by content
because the workbooks use three incompatible layouts across the sample.

In [ ]:
from src.data.cse_market_data import build_market_dataframe, load_all_yearly_security_files, load_aspi_index
from src.data.emdat_disasters import load_emdat
from src.data.macro_indicators import load_worldbank_macro, load_sp500_global_control

# Primary CSE market series: real local archive only (see \u00a76 for why no live gap-fill)
market = build_market_dataframe(RAW_DATA_DIR)[["date", "aspi_close", "trading_volume", "price_source", "volume_source"]]
print(f"Market series: {market.shape[0]} trading days, {market['date'].min().date()} -> {market['date'].max().date()}")
market.head()


### 1.3.2 Disaster records

Loads the EM-DAT export and maps its native schema onto the columns the feature
builder expects. Records outside the market series' coverage are loaded and counted
here, then dropped by stage 02's scope filter, so the loss is visible rather than
silent.

In [ ]:
# Real EM-DAT disaster export
# Upgraded 2026-09-12 to the 1986-2025 export (110 records, was 86 covering 2000-2025).
# The extra 24 records are all pre-2000. They are loaded here but CANNOT be modelled: the
# ASPI series starts 2000-01-03, and a day-0 event study needs the index on the event day
# plus 90 trading days after it. Stage 02's scope filter drops them, and the count is
# printed below so the loss is visible rather than silent.
EMDAT_PATH = EMDAT_DISASTERS_FILE
disasters_raw = load_emdat(EMDAT_PATH)
print(f"Raw EM-DAT records: {disasters_raw.shape[0]} "
      f"({disasters_raw['event_date'].min().date()} -> {disasters_raw['event_date'].max().date()})")
print(f"  pre-2000 (no ASPI coverage, cannot be modelled): "
      f"{(disasters_raw['event_date'] < '2000-01-01').sum()}")
print()
print("damage_source breakdown (how much of the set has a real vs. missing damage figure):")
print(disasters_raw["damage_source"].value_counts().to_string())
print()
# EM-DAT leaves Start Day blank for slow-onset events. Filling it with 1 invents a
# first-of-month event date, which for a DAY-0 study means scoring the market on a day the
# disaster did not begin. Recorded, not hidden -- the affected events are droughts, which
# have no meaningful day-0 at all.
print("event-date precision:")
print(disasters_raw["event_date_precision"].value_counts().to_string())
print()
print(f"Magnitude (physical, unit varies by hazard): "
      f"{disasters_raw['mag_area_km2'].notna().sum()} flood-area km2, "
      f"{disasters_raw['mag_wind_kph'].notna().sum()} storm-wind kph")
disasters_raw.head()

### 1.3.3 Macroeconomic controls

Annual World Bank series for Sri Lanka. The dating matters: a year's figures are not
published until the following year, so they are dated accordingly. Dating them to 1
January would hand an event the full-year numbers that summarise the year it occurred
in, look-ahead across four features.

In [ ]:
# Macro controls: World Bank (real, thesis-named source)
macro = load_worldbank_macro(start_year=2000, end_year=2025)
# Annual World Bank figures for year Y are dated Y+1-07-01, not Y-01-01.
# Dating them 1 January made them available to events that occurred DURING the
# year they summarize: a November 2005 event was being handed full-year 2005 GDP
# and inflation, numbers not published until well into 2006. That is look-ahead,
# and it contaminated 4 features (gdp_growth_pct, inflation_cpi_pct,
# gdp_current_usd, and damage_to_gdp which is derived from the last of these).
# Mid-year of the following year approximates the World Bank release schedule.
macro["date"] = pd.to_datetime((macro["year"] + 1).astype(str) + "-07-01")

# Reproducibility caveat, stated rather than implied: this series is pulled LIVE
# from the World Bank API, and its historical values are revised retrospectively.
# The figures below are the revision current as of the run date printed above --
# not the vintage that would have been observable at each event date. Re-running
# this notebook in a later year will produce slightly different macro inputs and
# therefore slightly different model metrics.
print("World Bank macro series (annual):")
macro


### 1.3.4 Global market control

Daily S&P 500 log returns, the thesis's named control for global market conditions.

In [ ]:
# Global control: S&P 500 (real, thesis-named source)
sp500 = load_sp500_global_control()
print(f"S&P 500 daily log returns: {sp500.shape[0]} rows, {sp500['date'].min().date()} -> {sp500['date'].max().date()}")
sp500.head()


## 1.4 Cleaning

**What.** Forward fill the gaps in the daily market series, so that a session with no
recorded value carries the last value actually observed.

**Why.** The exchange is thinly traded, and a missing daily volume is not the same thing as
a volume of zero. Filling a gap with zero would fabricate a liquidity collapse that never
happened, and the volume crash magnitude target is measured directly against a trailing
volume baseline, so the fabrication would land inside the target itself.

**Later use.** The cleaned series is what stage 02 measures all three targets from, and what
the daily market features are computed on.

**Justification.** Forward filling is the standard chronology safe treatment for a gap in a
financial series: it uses only information that existed before the gap, so it cannot leak a
future value backwards. Mean imputation, by contrast, would use the whole sample, including
sessions after the gap. The project's methodology rules out mean imputation for financial
series for exactly that reason.


In [ ]:
from src.data.preprocessing import forward_fill_missing_values

market_clean = market.set_index("date")
market_clean = forward_fill_missing_values(market_clean).reset_index()
print(f"Missing aspi_close after ffill: {market_clean['aspi_close'].isna().sum()}")
print(f"Missing trading_volume after ffill: {market_clean['trading_volume'].isna().sum()} "
      f"(real gaps remain where the archive itself has no volume data yet -- 2000, and post Mar-2023)")


## 1.5 Extending the ASPI series beyond the archive

The local archive's price series stops at **2023-06-28**, which excluded ten qualifying
EM-DAT events, including both Storm Ditwah dates, the thesis's own flagship example.
`countryeconomy.com` publishes the daily ASPI close and covers the gap.

The archive stays authoritative for every day it covers: `extend_market_series` appends
only rows strictly after the cutoff, so no existing value is ever overwritten. The 21-day
overlap is checked below as a continuity test rather than assumed.

`trading_volume` is left NaN on the appended rows, countryeconomy publishes the index
level only, and no practical post-2023 volume source was found (the CSE daily PDFs are
reachable but the index API returns only the five most recent, at ~2.7 MB each). The
existing per-target NaN mask therefore drops these events from Y2, exactly as it already
does for the 2000 volume gap.

In [ ]:
from pathlib import Path

from src.data.external_sources import extend_market_series, fetch_countryeconomy_aspi

EXTERNAL_CACHE = ARTIFACT_DIR / "external"
EXTERNAL_CACHE.mkdir(parents=True, exist_ok=True)

_archive_end = market_clean["date"].max()
_months = [f"{y}-{m:02d}" for y in range(_archive_end.year, 2027) for m in range(1, 13)]
_months = [m for m in _months if m >= _archive_end.strftime("%Y-%m")]

aspi_ext = fetch_countryeconomy_aspi(EXTERNAL_CACHE, _months)

# Continuity test on the overlap: if the external source disagreed with the archive on
# the days both cover, splicing them would introduce a level break at the join.
_overlap = market_clean.merge(aspi_ext, on="date", suffixes=("_arch", "_ext"))
_diff = (_overlap["aspi_close_arch"] - _overlap["aspi_close_ext"]).abs()
print(f"Overlap days: {len(_overlap)}  |  exact matches: {int((_diff < 0.005).sum())}"
      f"  |  max abs diff: {_diff.max():.2f}  |  mean: {_diff.mean():.3f}")
assert _diff.max() < 100, "External ASPI disagrees with the archive -- do not splice."

market_clean = extend_market_series(market_clean, aspi_ext)
print(f"Market series extended: {_archive_end.date()} -> {market_clean['date'].max().date()} "
      f"(+{(market_clean['date'] > _archive_end).sum()} trading days)")

## 1.6 Cache the acquired data

**What.** Write the four acquired tables to the artifact cache, each with a provenance
sidecar.

**Why.** Everything above is expensive, and none of it depends on any modelling choice, so
it is produced once and reloaded by every later stage rather than repeated.

**Later use.** Stage 02 reads all four tables and produces the single event level table that
the modelling stages consume.

**Justification.** Two of these sources are pulled live and unpinned, so a rerun months later
can return revised figures. The sidecar records the retrieval timestamp and the version of
every library involved, which is the only way a later reader can tell whether a number moved
because the method changed or because the source was revised.


In [ ]:
save_frame(market_clean, "market", "Daily ASPI close + summed market volume, forward-filled.")
save_frame(disasters_raw, "disasters", "Raw EM-DAT export, all records before scope filtering.")
save_frame(macro, "macro", "World Bank annual series, publication-lag corrected.")
save_frame(sp500, "sp500", "S&P 500 daily log return, global control.")


## 1.7 Final Outputs

This stage turns the raw archive and the live sources into four cached tables.

1. Cached tables written to `artifacts/tables/`: `market.parquet` (daily ASPI close and
   summed market volume), `disasters.parquet` (EM-DAT records with damage provenance),
   `macro.parquet` (World Bank annual controls) and `sp500.parquet` (global control).
2. Cached external downloads written to `artifacts/external/`: NASA POWER district
   weather, DesInventar losses, the FRED exchange rate series, the election calendar and
   the ASPI extension used beyond the local archive.
3. Every table carries a `.provenance.json` sidecar naming the retrieval time and the
   library versions, because several of these sources are pulled live and unpinned.
4. No model is fitted here and no figure is produced.
5. Consumed next by `02_features_targets.ipynb`, which builds the features and the three
   research targets from `market`, `disasters`, `macro` and `sp500`.

Execution assumptions: the raw workbooks are present in `data/raw/`, and network access is
available for the live sources. Re run this stage only when the raw data changes, because
the live sources can return revised figures that would move every downstream number.
